# Lab L4.2 · Deep learning trio — MLP credit challenger, CNN document router and LSTM cash forecaster
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 4 · ⏱ 100 min · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Three architectures, three data types, one honest question each: does deep learning beat the classical model we already have?

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the 💡 hint in the HTML lab guide, then ask your trainer. The Level 3 solution is released after the lab.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

**Part A — Credit:** Model Risk wants a neural *challenger* to the L3.1 logistic PD model. **Part B — Operations:** 15,000 scanned onboarding documents a month are routed by hand; a CNN could route them automatically. **Part C — Treasury:** can an LSTM beat the classical forecasting baselines for next-day network cash demand?

**Progressive architecture:** benchmarks come from L3.1 and L2.2; models are saved for Day 5.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Part A · Prepare the credit matrices and benchmark

**Why this matters.** Neural networks need dense, scaled float32 arrays — exactly what the L1.2 preprocessor produces. The benchmark to beat is the L3.1 logistic model’s validation AUC.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(RNG)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

In [ ]:
import joblib, json
from sklearn.base import clone
feat = pd.read_parquet(CKPT / "loan_features.parquet")
spec = json.loads((CKPT / "loan_feature_spec.json").read_text())
preprocessor = joblib.load(CKPT / "loan_preprocessor.joblib")      # fitted on train in Lab L1.2
TARGET = spec["target"]
train, valid, test = (feat[feat.split == s] for s in ["train", "valid", "test"])
print({s: len(d) for s, d in zip(["train", "valid", "test"], [train, valid, test])}, "| honest baseline AUC from L1.2:", spec["honest_valid_auc"])
from sklearn.metrics import roc_auc_score, average_precision_score
Xtr, Xva, Xte = (preprocessor.transform(d).astype("float32") for d in (train, valid, test))
ytr, yva, yte = (d[TARGET].to_numpy() for d in (train, valid, test))
logit = joblib.load(CKPT / "credit_model.joblib")
bench_auc = roc_auc_score(yva, logit.predict_proba(valid)[:, 1])
results = {"credit": {"logistic_valid_auc": bench_auc}}
print(Xtr.shape, "| logistic benchmark valid AUC:", round(bench_auc, 4))

**✔ Checkpoint.** `(207705, 46)`; benchmark AUC ≈ 0.829.

## Step 2 · Part A · MLP challenger with dropout and early stopping

**Why this matters.** An MLP can learn feature interactions automatically. **Dropout** randomly silences neurons during training (regularisation); **EarlyStopping** watches validation AUC and restores the best epoch. We do *not* use class weights so that outputs stay calibrated PDs.

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — hidden-layer non-linearity
- **Blank 2** *(inline)* — binary output → probability of default
- **Blank 3** *(inline)* — loss for a 0/1 target
- **Blank 4** *(inline)* — stop when validation AUC stops improving

In [ ]:
mlp = keras.Sequential([
    keras.Input(shape=(Xtr.shape[1],)),
    layers.Dense(64, activation=____), layers.Dropout(0.2),  # ✏️ B1
    layers.Dense(32, activation="relu"), layers.Dropout(0.2),
    layers.Dense(1, activation=____)])  # ✏️ B2
mlp.compile(optimizer=keras.optimizers.Adam(1e-3), loss=____, metrics=[keras.metrics.AUC(name="auc")])  # ✏️ B3
stop = ____  # ✏️ B4
hist = mlp.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=25, batch_size=1024, callbacks=[stop], verbose=0)
pd.DataFrame(hist.history)[["loss", "val_loss", "auc", "val_auc"]].plot(subplots=True, layout=(2, 2), figsize=(12, 6), marker="o"); plt.show()
p_mlp = mlp.predict(Xva, verbose=0).ravel()
results["credit"].update({"mlp_valid_auc": roc_auc_score(yva, p_mlp), "mlp_valid_pr_auc": average_precision_score(yva, p_mlp),
                          "mlp_epochs": len(hist.history["loss"]), "mlp_params": mlp.count_params()})
print({k: round(v, 4) if isinstance(v, float) else v for k, v in results["credit"].items()})

**✔ Checkpoint.** Training stops after ≈ 10–20 epochs. MLP AUC ≈ 0.828 vs logistic 0.829 — a tie, with ≈ 5,000 parameters instead of 46.

## Step 3 · Part B · Load images and build a dense-only baseline

**Why this matters.** To *prove* convolution helps, first train a network that ignores spatial structure: flatten the 64×64 pixels into 4,096 numbers and use Dense layers. Images are scaled to [0, 1] and split stratified 70/15/15.

#### ✏️ What to write in the next cell
- **Blank 5** *(inline)* — integer labels 0..4, multi-class

In [ ]:
from sklearn.model_selection import train_test_split
docs = np.load(DATA / "kyc_documents_64.npz")
X_img = (docs["X"].astype("float32") / 255.0)[..., None]; y_img = docs["y"]; CLASSES = list(docs["classes"])
Xi_tr, Xi_tmp, yi_tr, yi_tmp = train_test_split(X_img, y_img, test_size=0.30, stratify=y_img, random_state=RNG)
Xi_va, Xi_te, yi_va, yi_te = train_test_split(Xi_tmp, yi_tmp, test_size=0.50, stratify=yi_tmp, random_state=RNG)
print(Xi_tr.shape, Xi_va.shape, Xi_te.shape)
fig, axs = plt.subplots(1, 5, figsize=(12, 2.6))
for k, ax in enumerate(axs): ax.imshow(Xi_tr[yi_tr == k][0, ..., 0], cmap="gray"); ax.set_title(CLASSES[k], fontsize=9); ax.axis("off")
plt.show()
dense = keras.Sequential([keras.Input(shape=(64, 64, 1)), layers.Flatten(), layers.Dense(128, activation="relu"), layers.Dense(5, activation="softmax")])
dense.compile(optimizer="adam", loss=____, metrics=["accuracy"])  # ✏️ B5
dense.fit(Xi_tr, yi_tr, validation_data=(Xi_va, yi_va), epochs=8, batch_size=128, verbose=0)
results["documents"] = {"dense_test_acc": float(dense.evaluate(Xi_te, yi_te, verbose=0)[1]), "dense_params": dense.count_params()}
results["documents"]

**✔ Checkpoint.** (10500, 64, 64, 1) train images. The dense baseline reaches only ≈ 65–70% test accuracy with ≈ 525k parameters — documents appear at different positions, scales and angles, and a dense layer must re-learn each position separately.

## Step 4 · Part B · CNN with augmentation

**Why this matters.** Convolutions share small filters across the image — far fewer parameters, and **translation-invariant** features (a seal is a seal anywhere on the page). **Pooling** shrinks feature maps; **GlobalAveragePooling** replaces a huge flatten layer; **augmentation** (small rotations/shifts) mimics real scanning variation.

#### ✏️ What to write in the next cell
- **Blank 6** *(inline)* — halve the spatial size
- **Blank 7** *(inline)* — one number per feature map — tiny head

In [ ]:
augment = keras.Sequential([layers.RandomRotation(0.03), layers.RandomTranslation(0.05, 0.05)], name="augment")
cnn = keras.Sequential([
    keras.Input(shape=(64, 64, 1)), augment,
    layers.Conv2D(16, 3, activation="relu", padding="same"), ____,  # ✏️ B6
    layers.Conv2D(32, 3, activation="relu", padding="same"), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    ____,  # ✏️ B7
    layers.Dense(64, activation="relu"), layers.Dropout(0.3), layers.Dense(5, activation="softmax")])
cnn.compile(optimizer=keras.optimizers.Adam(2e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
t0 = time.perf_counter()
cnn.fit(Xi_tr, yi_tr, validation_data=(Xi_va, yi_va), epochs=25, batch_size=64, verbose=0,
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=5, restore_best_weights=True)])
results["documents"].update({"cnn_test_acc": float(cnn.evaluate(Xi_te, yi_te, verbose=0)[1]), "cnn_params": cnn.count_params(), "cnn_train_s": time.perf_counter() - t0})
print(results["documents"])

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
pred_i = cnn.predict(Xi_te, verbose=0).argmax(1)
ConfusionMatrixDisplay.from_predictions(yi_te, pred_i, display_labels=CLASSES, normalize="true", values_format=".2f", cmap="Blues", xticks_rotation=30); plt.show()
wrong = np.where(pred_i != yi_te)[0][:6]
fig, axs = plt.subplots(1, max(len(wrong), 1), figsize=(13, 2.6))
for ax, i in zip(np.atleast_1d(axs), wrong): ax.imshow(Xi_te[i, ..., 0], cmap="gray"); ax.set_title(f"true {CLASSES[yi_te[i]]}\npred {CLASSES[pred_i[i]]}", fontsize=8); ax.axis("off")
plt.show()

**✔ Checkpoint.** CNN test accuracy ≈ 96–98% with ≈ 28k parameters (≈ 19× fewer than the dense model) — convolution + pooling give translation-tolerant features. Remaining errors are small, smudged or heavily rotated scans, often statement ↔ invoice (both tabular).

## Step 5 · Part C · Sequences for next-day cash with known-in-advance features

**Why this matters.** An LSTM reads a window of past days. To be fair to SARIMAX (which knew the calendar), each timestep carries that day’s **log withdrawal** plus the **next day’s** calendar flags — the information Cash Ops genuinely has the evening before. We standardise with **training** statistics only.

#### ✏️ What to write in the next cell
- **Blank 8** *(code block, ≈2 lines)* — build Xs (samples, W, n_features) from windows F[i:i+W] and ys = z[i+W] (next day), plus dates = net.index[i+W]

In [ ]:
from sklearn.linear_model import Ridge
net = pd.read_parquet(CKPT / "atm_network_daily.parquet").loc["2022-01-01":]
FLAGS = ["is_payday_window", "is_pre_eid_window", "is_public_holiday", "is_weekend", "is_ramadan"]
logy = np.log(net.withdrawal_aed)
test_start = pd.Timestamp("2025-07-03")
mu, sd = logy[logy.index < test_start].mean(), logy[logy.index < test_start].std()      # training statistics only
z = ((logy - mu) / sd).to_numpy("float32")
nextcal = net[FLAGS].shift(-1).fillna(0).to_numpy("float32")                            # tomorrow's calendar, known today
F = np.column_stack([z, nextcal])
W = 28
# ✏️ BLANK 8 — write ≈2 line(s): build Xs (samples, W, n_features) from windows F[i:i+W] and ys = z[i+W] (next day), plus dates = net.index[i+W]
# ...your code here...
is_te = dates >= test_start
Xs_tr, Xs_te, ys_tr, ys_te = Xs[~is_te], Xs[is_te], ys[~is_te], ys[is_te]
to_aed = lambda v: np.exp(v * sd + mu)
actual = to_aed(ys_te)
wape = lambda a, f: np.abs(a - f).sum() / a.sum()
results["cash"] = {"seasonal_naive_wape": wape(actual, to_aed(Xs_te[:, -7, 0]))}         # value 7 days before the target
ridge_seq = Ridge(alpha=1.0).fit(Xs_tr.reshape(len(Xs_tr), -1), ys_tr)
results["cash"]["ridge_window_wape"] = wape(actual, to_aed(ridge_seq.predict(Xs_te.reshape(len(Xs_te), -1))))
print(Xs.shape, "| train", Xs_tr.shape[0], "test", Xs_te.shape[0], results["cash"])

**✔ Checkpoint.** Tensor ≈ (1372, 28, 6); 121 test days. Seasonal naive WAPE ≈ 12% (it ignores the calendar); Ridge on windows with next-day flags ≈ 1.3%.

## Step 6 · Part C · Train the LSTM and compare

**Why this matters.** The LSTM’s gates learn what to remember across the 28 days. With only ~1,250 training sequences it can over-fit, so we hold out the last 10% of training windows (chronologically) for early stopping.

#### ✏️ What to write in the next cell
- **Blank 9** *(inline)* — recurrent layer with 32 memory units
- **Blank 10** *(inline)* — regression loss on the standardised target

In [ ]:
lstm = keras.Sequential([keras.Input(shape=Xs.shape[1:]), ____, layers.Dense(16, activation="relu"), layers.Dense(1)])  # ✏️ B9
lstm.compile(optimizer=keras.optimizers.Adam(3e-3), loss=____)  # ✏️ B10
h = lstm.fit(Xs_tr, ys_tr, validation_split=0.1, epochs=60, batch_size=32, verbose=0,
             callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)])
f_lstm = to_aed(lstm.predict(Xs_te, verbose=0).ravel())
results["cash"].update({"lstm_wape": wape(actual, f_lstm), "lstm_epochs": len(h.history["loss"])})
print({k: round(v, 4) if isinstance(v, float) else v for k, v in results["cash"].items()})
plt.figure(figsize=(13, 4)); plt.plot(dates[is_te], actual / 1e6, "k", lw=2, label="actual")
plt.plot(dates[is_te], f_lstm / 1e6, label="LSTM"); plt.plot(dates[is_te], to_aed(ridge_seq.predict(Xs_te.reshape(len(Xs_te), -1))) / 1e6, "--", label="Ridge window")
plt.ylabel("AED m"); plt.legend(); plt.title("Next-day network cash demand — test period"); plt.show()

In [ ]:
mlp.save(CKPT / "credit_mlp.keras"); cnn.save(CKPT / "kyc_cnn.keras"); lstm.save(CKPT / "atm_lstm.keras")
clean = lambda d: {k: (float(v) if isinstance(v, (float, np.floating)) else int(v)) for k, v in d.items()}
(CKPT / "dl_results.json").write_text(json.dumps({k: clean(v) for k, v in results.items()}, indent=2))
print(json.dumps({k: clean(v) for k, v in results.items()}, indent=2))

**✔ Checkpoint.** LSTM WAPE ≈ 1.4–2% — far better than seasonal naive, but *not* better than the Ridge window model (≈ 1.3%). With ~1,250 sequences and strong calendar features, the linear model is hard to beat.

## ✅ Lab complete
You can now say *with evidence* where deep learning earns its complexity. **Tomorrow:** taking models to production — and keeping them healthy.